### Data Ingestion

In [2]:
###document data structure

from langchain_core.documents import Document

In [3]:
doc=Document(
    page_content="This is the content of the document. I am using create a RAG",
    metadata={"source": "example.txt", "author": "John Doe", "pages": 1, "date_created": "2024-06-01" }
)
doc

Document(metadata={'source': 'example.txt', 'author': 'John Doe', 'pages': 1, 'date_created': '2024-06-01'}, page_content='This is the content of the document. I am using create a RAG')

In [4]:
## create a simple txt file

import os
os.makedirs("../data/text_files", exist_ok=True)

In [5]:
sample_texts ={ "../data/text_files/sample.txt": """Introduction to Python
----------------------

Python is a high-level, interpreted programming language known for its simple syntax and readability.
It is widely used in web development, artificial intelligence, automation, data science,
machine learning, backend development, and scripting.

Python was created by Guido van Rossum and released in 1991.
It helps developers write clean and efficient code with fewer lines compared to many other languages.


Features of Python
------------------

1. Easy to Learn and Read
Python has simple and beginner-friendly syntax.

2. Interpreted Language
Python code executes line by line, making debugging easier.

3. Cross-Platform
Python works on Windows, macOS, and Linux.

4. Large Standard Library
Python provides built-in modules for many tasks.

5. Object-Oriented
Python supports object-oriented programming concepts.

6. Huge Community Support
Millions of developers contribute libraries, tutorials, and tools.

7. Used in AI and Data Science
Python is heavily used in machine learning and artificial intelligence.

8. Fast Development
Developers can build applications quickly using Python.
"""
}
# Create TXT file

for filepath, content in sample_texts.items():
    with open(filepath, "w", encoding="utf-8") as file:
        file.write(content)


print("sample.txt file created successfully!")

sample.txt file created successfully!


In [8]:
### Read the text using the text loader


from langchain_community.document_loaders import TextLoader

loader = TextLoader("../data/text_files/sample.txt")
document = loader.load()
print(document)

[Document(metadata={'source': '../data/text_files/sample.txt'}, page_content='Introduction to Python\n----------------------\n\nPython is a high-level, interpreted programming language known for its simple syntax and readability.\nIt is widely used in web development, artificial intelligence, automation, data science,\nmachine learning, backend development, and scripting.\n\nPython was created by Guido van Rossum and released in 1991.\nIt helps developers write clean and efficient code with fewer lines compared to many other languages.\n\n\nFeatures of Python\n------------------\n\n1. Easy to Learn and Read\nPython has simple and beginner-friendly syntax.\n\n2. Interpreted Language\nPython code executes line by line, making debugging easier.\n\n3. Cross-Platform\nPython works on Windows, macOS, and Linux.\n\n4. Large Standard Library\nPython provides built-in modules for many tasks.\n\n5. Object-Oriented\nPython supports object-oriented programming concepts.\n\n6. Huge Community Suppor

In [9]:
## Directory loader

from langchain_community.document_loaders import DirectoryLoader

## load all the text files in the directory
dir_loader = DirectoryLoader("../data/text_files", glob="*.txt" , loader_cls=TextLoader, show_progress=False, loader_kwargs={"encoding": "utf-8"})
documents = dir_loader.load()
documents

[Document(metadata={'source': '../data/text_files/sample.txt'}, page_content='Introduction to Python\n----------------------\n\nPython is a high-level, interpreted programming language known for its simple syntax and readability.\nIt is widely used in web development, artificial intelligence, automation, data science,\nmachine learning, backend development, and scripting.\n\nPython was created by Guido van Rossum and released in 1991.\nIt helps developers write clean and efficient code with fewer lines compared to many other languages.\n\n\nFeatures of Python\n------------------\n\n1. Easy to Learn and Read\nPython has simple and beginner-friendly syntax.\n\n2. Interpreted Language\nPython code executes line by line, making debugging easier.\n\n3. Cross-Platform\nPython works on Windows, macOS, and Linux.\n\n4. Large Standard Library\nPython provides built-in modules for many tasks.\n\n5. Object-Oriented\nPython supports object-oriented programming concepts.\n\n6. Huge Community Suppor

### Embeddings and the VectorStore DB

In [10]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

/Users/devanshsharma/Documents/RAG/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
class EmbeddingManager:
    """ Handles document embedding generation using sentence transformers. """
    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """ Initializes the embedding model. 
        Args:
        model_name : Hugging Face model name for sentence embeddings.
        """
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        """ Loads the sentence transformer model. """
        try:
            print(f"Loading model '{self.model_name}'...")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model '{self.model_name}' loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model '{self.model_name}': {e}")

            raise

    def generate_embeddings(self, documents: List[Document]) -> List[np.ndarray]:
        """ Generates embeddings for a list of documents. 
        Args:
        documents : List of Document objects to embed.
        Returns:
        List of numpy arrays containing the embeddings.
        """
        if not self.model:
            raise ValueError("Embedding model is not loaded.")
        
        print (f"Generating embeddings for {len(documents)} texts...")   
        embeddings = self.model.encode([doc.page_content for doc in documents], show_progress_bar=True)
        print("Embeddings generated successfully with shape:", embeddings.shape)
        return embeddings
    

    ### initialize the embedding manager and generate embeddings for the documents
embedding_manager = EmbeddingManager()
embedding_manager

AttributeError: 'EmbeddingManager' object has no attribute 'load_model'